# Week 4 — Predictive Modeling and Optimization in Logistics

**YuvaIntern | Logistics Data Analyst Intern**

This notebook develops a predictive modeling and optimization workflow for a hypothetical logistics operation. The target variable is delivery time in days.

**Important:** The dataset is synthetic. Model performance and optimization results are illustrative and should not be interpreted as real operational performance.

## 1. Prediction Problem

### Target
`Delivery_Time_Days`

### Candidate predictors
- Transport mode
- Region
- Priority
- Shipment volume
- Route distance
- Warehouse processing time
- Weather disruption
- Peak-period status
- Transport cost

The objective is to estimate delivery time before the shipment is completed and demonstrate how predictions can feed into an operational optimization scenario.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('data/hypothetical_logistics_week4_dataset.csv')
df.head()

## 2. Dataset Inspection

In [ ]:
print('Shape:', df.shape)
display(df.describe(include='all'))
display(df.isna().sum())

## 3. Train/Test Split and Preprocessing

Categorical variables are one-hot encoded. Numerical variables are passed through without scaling because the selected tree-based models do not require it, while the linear regression model can still operate on these features.

In [ ]:
target = 'Delivery_Time_Days'
X = df.drop(columns=[target, 'Delayed'])
y = df[target]

categorical = ['Transport_Mode', 'Region', 'Priority']
numerical = [
    'Shipment_Volume', 'Distance_KM', 'Warehouse_Processing_Hours',
    'Weather_Disruption', 'Peak_Period', 'Transport_Cost'
]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', 'passthrough', numerical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

## 4. Model Selection

Four regression approaches are compared:

- Linear Regression — interpretable baseline
- Decision Tree Regression — captures non-linear decision rules
- Random Forest Regression — ensemble of decision trees
- Gradient Boosting Regression — sequential ensemble model

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=8),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=200, max_depth=12, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42, n_estimators=150)
}

results = []
predictions = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predictions[name] = pred
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values('RMSE')
results_df

## 5. Model Evaluation

- **MAE:** average absolute prediction error in days.
- **RMSE:** penalizes larger errors more strongly.
- **R²:** proportion of target variance explained by the model.

The synthetic test-set results can be inspected below.

In [ ]:
results_df.round(3)

## 6. Five-Fold Cross-Validation

Cross-validation provides an additional view of model stability across different training/validation splits.

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_estimators=200, max_depth=12, n_jobs=-1))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse = np.sqrt(-cross_val_score(
    rf_pipeline, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1
))

print('Fold RMSE:', np.round(cv_rmse, 3))
print('Mean RMSE:', round(cv_rmse.mean(), 3))

## 7. Random Forest Hyperparameter Tuning

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 12, None],
    'model__min_samples_split': [2, 5]
}

rf_base = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

grid = GridSearchCV(
    rf_base, param_grid, cv=5,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)

tuned_pred = grid.predict(X_test)
print('Best parameters:', grid.best_params_)
print('Tuned RMSE:', round(np.sqrt(mean_squared_error(y_test, tuned_pred)), 3))
print('Tuned MAE:', round(mean_absolute_error(y_test, tuned_pred), 3))
print('Tuned R2:', round(r2_score(y_test, tuned_pred), 3))

## 8. Actual vs Predicted

The closest points to the diagonal line indicate predictions close to actual delivery times.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pred = predictions[best_model_name]

plt.figure(figsize=(7,6))
plt.scatter(y_test, best_pred, alpha=0.55)
lo = min(y_test.min(), best_pred.min())
hi = max(y_test.max(), best_pred.max())
plt.plot([lo, hi], [lo, hi])
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.xlabel('Actual Delivery Time (Days)')
plt.ylabel('Predicted Delivery Time (Days)')
plt.show()

## 9. Optimization Scenario

The predictive model is connected to a simple transport-selection problem. A representative shipment is evaluated under Road, Rail, and Air alternatives.

The objective combines estimated transport cost with a penalty for predicted delivery time exceeding a 5-day service target.

$$Objective = Transport\ Cost + 75 \times max(Predicted\ Days - 5, 0)$$

In [ ]:
scenario = pd.DataFrame({
    'Transport_Mode': ['Road', 'Rail', 'Air'],
    'Region': ['Central'] * 3,
    'Priority': ['Standard'] * 3,
    'Shipment_Volume': [250] * 3,
    'Distance_KM': [1000] * 3,
    'Warehouse_Processing_Hours': [8] * 3,
    'Weather_Disruption': [0] * 3,
    'Peak_Period': [0] * 3,
    'Transport_Cost': [380, 520, 900]
})

best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', models[best_model_name])
])
best_pipe.fit(X_train, y_train)
scenario['Predicted_Delivery_Days'] = best_pipe.predict(scenario)
scenario['Late_Days'] = np.maximum(scenario['Predicted_Delivery_Days'] - 5, 0)
scenario['Objective_Cost'] = scenario['Transport_Cost'] + scenario['Late_Days'] * 75
scenario

## 10. Operational Interpretation

This optimization example demonstrates how a prediction can become an input to a decision-support process. In a real logistics system, the objective would need additional constraints such as vehicle capacity, route availability, service-level agreements, inventory requirements, fuel costs, and operational schedules.

Because the current data is synthetic, the numerical output is illustrative rather than an operational recommendation.

## 11. Limitations

- The dataset is synthetic.
- Transport costs are simulated.
- The optimization objective is intentionally simplified.
- The model does not represent a deployed logistics system.
- Correlations and model performance may differ substantially on real operational data.
- A production system would require monitoring, validation, drift detection, and business constraints.

## 12. Conclusion

The Week 4 workflow demonstrates the transition from descriptive analytics to predictive and decision-oriented analytics. Multiple regression models are compared using MAE, RMSE, and R²; cross-validation and hyperparameter tuning are demonstrated; and a simple transport-mode optimization scenario connects predicted delivery time with cost and service-level considerations.

This completes the four-week progression from strategic planning and preprocessing through EDA, predictive modeling, and optimization.